# AeroTwin AI – Notebook 01

**Objetivo:** Construir un gemelo digital (digital twin) del ecosistema aeronáutico español utilizando datos abiertos de AENA y ENAIRE junto con técnicas de IA generativa (RAG).

**Fuentes de datos:**
- AENA – estadísticas de pasajeros (`data/raw/aena_passengers/`)
- AENA – directorio de aerolíneas (`data/raw/aena_airlines/`)
- AENA – metadatos de aeropuertos (`data/raw/aena_airports/`)
- ENAIRE – publicación de información aeronáutica AIP (`data/raw/enaire_aip/`)

## 0. Configuración del entorno

In [ ]:
import os
import sys
from pathlib import Path

# Añadir el directorio raíz al path para importar src/
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Cargar variables de entorno desde .env (si existe)
from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=False)

print("ROOT:", ROOT)
print("OPENAI_API_KEY configurada:", bool(os.getenv("OPENAI_API_KEY")))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Carga de datos

In [ ]:
from src.loaders import (
    load_aena_passengers,
    load_aena_airlines,
    load_aena_airports,
    load_enaire_aip_texts,
)

# Descomentar cuando los datos estén disponibles en data/raw/
# df_passengers = load_aena_passengers()
# df_airlines   = load_aena_airlines()
# df_airports   = load_aena_airports()
# aip_docs      = load_enaire_aip_texts()

print("Loaders importados correctamente.")

## 2. Análisis exploratorio de datos (EDA)

> Ejecutar esta sección una vez que los datos estén disponibles en `data/raw/`.

In [ ]:
# Ejemplo de análisis exploratorio (completar con datos reales)
# print(df_passengers.shape)
# df_passengers.head()
print("EDA – pendiente de datos.")

## 3. Indexación de documentos AIP (Vector Store)

Construimos un índice FAISS con embeddings de OpenAI sobre los documentos AIP de ENAIRE.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document

# Ejemplo: crear documentos LangChain desde los textos AIP
# raw_docs = load_enaire_aip_texts()
# lc_docs = [Document(page_content=d["content"], metadata={"source": d["source"]}) for d in raw_docs]

# splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# chunks = splitter.split_documents(lc_docs)

# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# vector_store = FAISS.from_documents(chunks, embeddings)
# vector_store.save_local(ROOT / "data" / "processed" / "faiss_aip_index")

print("Indexación – pendiente de documentos AIP en data/raw/enaire_aip/.")

## 4. Pipeline RAG – AeroTwin AI

Combinamos el contexto extraído del vector store con un LLM para responder preguntas sobre el ecosistema aeronáutico.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

SYSTEM_PROMPT = """
Eres AeroTwin AI, un asistente especializado en el ecosistema aeronáutico español.
Utiliza únicamente el contexto proporcionado para responder las preguntas.
Si no conoces la respuesta, di que no tienes información suficiente.

Contexto:
{context}

Pregunta: {question}
Respuesta:"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=SYSTEM_PROMPT,
)

# llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"), temperature=0)

# Cargar índice guardado
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# vector_store = FAISS.load_local(
#     ROOT / "data" / "processed" / "faiss_aip_index",
#     embeddings,
#     allow_dangerous_deserialization=True,
# )

# qa_chain = RetrievalQA.from_chain_type(
#     llm=llm,
#     chain_type="stuff",
#     retriever=vector_store.as_retriever(search_kwargs={"k": 5}),
#     chain_type_kwargs={"prompt": prompt},
# )

print("Pipeline RAG – pendiente de índice vectorial.")

## 5. Preguntas de demostración

Ejecuta las preguntas del fichero `outputs/demo_questions.md` contra el pipeline RAG.

In [ ]:
demo_questions = [
    "¿Cuál fue el aeropuerto español con mayor tráfico de pasajeros en 2023?",
    "¿Cuáles son los procedimientos de llegada instrumental (STAR) vigentes en Madrid-Barajas?",
    "¿Qué aerolíneas operan más rutas en los aeropuertos de AENA?",
]

for question in demo_questions:
    print(f"\n❓ {question}")
    # answer = qa_chain.run(question)
    # print(f"💡 {answer}")
    print("   → (pipeline RAG pendiente de datos)")

## 6. Próximos pasos

1. Descargar los datasets de AENA desde el [portal de datos abiertos](https://estadisticas.aena.es/estadisticas/SASPortal/main.do) y colocarlos en `data/raw/aena_passengers/`, `data/raw/aena_airlines/` y `data/raw/aena_airports/`.
2. Descargar los documentos AIP de ENAIRE desde [su portal oficial](https://aip.enaire.es/) y colocarlos en `data/raw/enaire_aip/`.
3. Completar la celda de indexación (sección 3) y ejecutarla para construir el índice FAISS.
4. Probar el pipeline RAG con las preguntas de demostración (sección 5).
5. Evaluar la calidad de las respuestas y ajustar los parámetros de recuperación.